In [1]:
%pip install -qU "langchain==1.3.11" langchain-openai langgraph langgraph-checkpoint langgraph-checkpoint-postgres faiss-cpu python-dotenv


Note: you may need to restart the kernel to use updated packages.


Key mapping from the old API to the new one:

Old (langchain <1.0 / langchain-classic)	New (LangChain v1.3.11)
langchain.chat_models.ChatOpenAI	langchain_openai.ChatOpenAI (or init_chat_model)
langchain.chains.LLMChain	plain prompt | llm Runnable, or a @tool function
langchain.agents.initialize_agent + AgentType	langchain.agents.create_agent
langchain.memory.ConversationBufferMemory	checkpointer=InMemorySaver() + thread_id (short-term memory in graph state)
langchain.memory.ConversationBufferWindowMemory	custom before_model middleware that trims the message window
langchain.memory.ConversationSummaryMemory	SummarizationMiddleware
langchain.memory.VectorStoreRetrieverMemory	langgraph.store.memory.InMemoryStore (semantic/long-term store) + a memory tool
PostgresChatMessageHistory	langgraph.checkpoint.postgres.PostgresSaver
Reference: What's new in LangChain v1 — create_agent

In [2]:
import os
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

# Load API keys
load_dotenv(".env")
openai_api_key = os.getenv("openai_api_key")

# LLM — langchain_openai.ChatOpenAI is the supported import in v1
# (langchain.chat_models.ChatOpenAI was removed; legacy chains moved to langchain-classic)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# Tool: Simple QA
# LLMChain is gone in v1 — a tool is now just a plain Python function decorated with @tool.
@tool
def simple_qa(question: str) -> str:
    """Answer factual questions clearly."""
    return llm.invoke(f"Answer clearly: {question}").content


1. Short-term memory (replaces ConversationBufferMemory)
In v1, an agent built with create_agent keeps the full running conversation as part of its graph state. Attaching a checkpointer persists that state across calls, and a thread_id selects which conversation to continue — this is the direct replacement for ConversationBufferMemory(memory_key="chat_history", return_messages=True) + initialize_agent(..., memory=memory).

In [3]:
# 1) Short-term memory via checkpointer (was: ConversationBufferMemory)
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[simple_qa],
    system_prompt="You are a helpful assistant. Use the simple_qa tool for factual questions.",
    checkpointer=checkpointer,
)

thread_config = {"configurable": {"thread_id": "buffer-memory-demo"}}

print("1\ufe0f\u20e3 First Question")
res1 = agent.invoke(
    {"messages": [{"role": "user", "content": "What is LangChain?"}]},
    thread_config,
)
print("\nAnswer:", res1["messages"][-1].content)

print("\n2\ufe0f\u20e3 Follow-up Question")
res2 = agent.invoke(
    {"messages": [{"role": "user", "content": "Who created it?"}]},
    thread_config,
)
print("\nAnswer:", res2["messages"][-1].content)

print("\n3\ufe0f\u20e3 Ask again about previous topic")
res3 = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain it simply again."}]},
    thread_config,
)
print("\nAnswer:", res3["messages"][-1].content)

# Inspect the full persisted conversation (equivalent to memory.chat_memory.messages)
for msg in res3["messages"]:
    print(f"{msg.type.upper()}: {msg.content}")


1️⃣ First Question

Answer: LangChain is a framework designed to help developers build applications that use large language models (LLMs) more effectively. It provides tools and components to manage prompts, chain together multiple calls to language models, integrate with external data sources, and handle memory and state, enabling the creation of complex, context-aware AI applications.

2️⃣ Follow-up Question

Answer: LangChain was created by Harrison Chase.

3️⃣ Ask again about previous topic

Answer: LangChain is a tool that helps programmers build smart apps using language models like me. It makes it easier to connect different parts, remember things, and use extra information, so the app can understand and respond better.
HUMAN: What is LangChain?
AI: 
TOOL: LangChain is a framework designed to help developers build applications that use large language models (LLMs) more effectively. It provides tools and components to manage prompts, chain together multiple calls to language mode

2. Windowed memory (replaces ConversationBufferWindowMemory(k=3))
There's no built-in "keep only the last k messages" memory class anymore. The recommended v1 pattern is a small custom middleware with a before_model hook that trims the message list right before it's sent to the model — the state itself (and the checkpoint) still holds everything, but the model only sees the trailing window.

In [4]:
# 2) Windowed memory (was: ConversationBufferWindowMemory(k=3))
from langchain.agents.middleware import AgentMiddleware
from langchain.messages import trim_messages

class WindowedMemoryMiddleware(AgentMiddleware):
    """Keep only the last `k` conversational turns visible to the model."""

    def __init__(self, k: int = 3):
        super().__init__()
        self.k = k

    def before_model(self, state, runtime):
        trimmed = trim_messages(
            state["messages"],
            max_tokens=self.k * 2,  # roughly k human/AI turn pairs
            token_counter=len,      # count by number of messages, not tokens
            strategy="last",
            start_on="human",
        )
        return {"messages": trimmed}

windowed_agent = create_agent(
    model=llm,
    tools=[simple_qa],
    system_prompt="You are a helpful assistant. Use the simple_qa tool for factual questions.",
    middleware=[WindowedMemoryMiddleware(k=3)],
    checkpointer=InMemorySaver(),
)

window_config = {"configurable": {"thread_id": "window-memory-demo"}}

print("1\ufe0f\u20e3 First Question")
res1 = windowed_agent.invoke(
    {"messages": [{"role": "user", "content": "What is LangChain?"}]},
    window_config,
)
print("\nAnswer:", res1["messages"][-1].content)

print("\n2\ufe0f\u20e3 Follow-up Question")
res2 = windowed_agent.invoke(
    {"messages": [{"role": "user", "content": "Who created it?"}]},
    window_config,
)
print("\nAnswer:", res2["messages"][-1].content)

print("\n3\ufe0f\u20e3 Ask again about previous topic")
res3 = windowed_agent.invoke(
    {"messages": [{"role": "user", "content": "Explain it simply again."}]},
    window_config,
)
print("\nAnswer:", res3["messages"][-1].content)

for msg in res3["messages"]:
    print(f"{msg.type.upper()}: {msg.content}")


1️⃣ First Question

Answer: LangChain is a framework designed to help developers build applications that use large language models (LLMs) more effectively. It provides tools and components to manage prompts, chain together multiple calls to LLMs, integrate with external data sources, and handle memory, enabling the creation of complex, context-aware, and interactive AI applications.

2️⃣ Follow-up Question

Answer: LangChain was created by Harrison Chase.

3️⃣ Ask again about previous topic

Answer: LangChain is a tool that helps programmers build smart apps using language models like me. It makes it easier to connect different parts, remember things, and use extra information, so the app can have better and more useful conversations.
HUMAN: What is LangChain?
AI: 
TOOL: LangChain is a framework designed to help developers build applications that use large language models (LLMs) more effectively. It provides tools and components to manage prompts, chain together multiple calls to LLMs,

3. Summarized memory (replaces ConversationSummaryMemory)
LangChain v1 ships a prebuilt SummarizationMiddleware for exactly this: once the conversation crosses a token threshold, older messages are collapsed into a running summary automatically, instead of you managing a separate summary LLM chain by hand.

In [5]:
#3.Summarized memory (replaces ConversationSummaryMemory)
from langchain.agents.middleware import SummarizationMiddleware

summarizing_agent = create_agent(
    model=llm,
    tools=[simple_qa],
    system_prompt="You are a helpful assistant. Use the simple_qa tool for factual questions.",
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger={"tokens": 500},  # summarize once history exceeds ~500 tokens
        ),
    ],
    checkpointer=InMemorySaver(),
)

summary_config = {"configurable": {"thread_id": "summary-memory-demo"}}

print("1\ufe0f\u20e3 First Question")
res1 = summarizing_agent.invoke(
    {"messages": [{"role": "user", "content": "What is LangChain?"}]},
    summary_config,
)
print("\nAnswer:", res1["messages"][-1].content)

print("\n2\ufe0f\u20e3 Follow-up Question")
res2 = summarizing_agent.invoke(
    {"messages": [{"role": "user", "content": "Who created it?"}]},
    summary_config,
)
print("\nAnswer:", res2["messages"][-1].content)

print("\n3\ufe0f\u20e3 Ask again about previous topic")
res3 = summarizing_agent.invoke(
    {"messages": [{"role": "user", "content": "Explain it simply again."}]},
    summary_config,
)
print("\nAnswer:", res3["messages"][-1].content)

for msg in res3["messages"]:
    print(f"{msg.type.upper()}: {msg.content}")


1️⃣ First Question

Answer: LangChain is a framework designed to help developers build applications that use large language models (LLMs) more effectively. It provides tools and components to manage prompts, chain together multiple calls to language models, integrate with external data sources, and handle memory and state, enabling the creation of complex, context-aware AI applications.

2️⃣ Follow-up Question

Answer: LangChain was created by Harrison Chase.

3️⃣ Ask again about previous topic

Answer: LangChain is a tool that helps programmers build smart apps using language models like me. It makes it easier to connect different parts, remember things, and use extra information, so the app can understand and respond better.
HUMAN: What is LangChain?
AI: 
TOOL: LangChain is a framework designed to help developers build applications that use large language models (LLMs) more effectively. It provides tools and components to manage prompts, chain together multiple calls to language mode